# Train the adaptive Q/R model on Colab

1. Set the parameters in the next cell (Google Drive links or file ids of the two zips).
2. Run all cells in order. Data is downloaded with `gdown` and unzipped under `/content/data`.

Expected zip contents (a single top folder inside a zip is fine):

```
mot-gt.zip      -> MOT17/{train,val}/<seq>/{gt/gt.txt, seqinfo.ini}, MOT20/..., DanceTrack/..., SportsMOT/...
```

Detections can be given either way (or both):

```
one zip:        detections.zip -> YOLOXx-ft/<Dataset>/<seq>.txt  and  YOLOXx-r/<Dataset>/<seq>.txt
one per folder: YOLOXx-ft.zip  -> <Dataset>/<seq>.txt            YOLOXx-r.zip -> <Dataset>/<seq>.txt
```

In [ ]:
# ---- parameters -------------------------------------------------------------
REPO_URL = "https://github.com/meysam-omidy/motion-predictor.git"
BRANCH = "paper-updates"

# Google Drive share link OR file id of each zip (anything gdown accepts).
GT_ZIP = "PUT_LINK_OR_ID_OF_mot-gt.zip"

# Detections, option A: one zip holding all detector folders ("" = not used).
DETECTIONS_ZIP = ""
# Detections, option B: one zip per detector folder ("" = not used / comes from option A).
DETECTION_FOLDER_ZIPS = {
    "YOLOXx-ft": "PUT_LINK_OR_ID_OF_YOLOXx-ft.zip",
    "YOLOXx-r": "PUT_LINK_OR_ID_OF_YOLOXx-r.zip",
}

# Detector folders to train on; all listed ones are aggregated.
DET_FOLDERS = ["YOLOXx-ft", "YOLOXx-r"]

# Save checkpoints to Google Drive so they survive a disconnect.
SAVE_TO_DRIVE = True
DRIVE_DIR = "/content/drive/MyDrive/motion-predictor-checkpoints"

DATA_DIR = "/content/data"
REPO_DIR = "/content/motion-predictor"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: Runtime > Change runtime type > GPU"

## 1. Code and requirements

In [ ]:
import os, subprocess
if os.path.isdir(f"{REPO_DIR}/.git"):
    !git -C {REPO_DIR} fetch -q origin && git -C {REPO_DIR} checkout -q {BRANCH} && git -C {REPO_DIR} pull -q origin {BRANCH}
else:
    !git clone -q -b {BRANCH} {REPO_URL} {REPO_DIR}
!git -C {REPO_DIR} log --oneline -1
%cd {REPO_DIR}

In [ ]:
# torch / numpy / pandas are preinstalled on Colab; these are the extras.
!pip install -q --upgrade gdown pytest
import torch, numpy, pandas
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "| numpy", numpy.__version__, "| pandas", pandas.__version__)

## 2. Data

In [ ]:
import glob, shutil, zipfile
import gdown

DATASETS = ("MOT17", "MOT20", "DanceTrack", "SportsMOT")

def fetch_zip(source, target):
    """Download a Drive link/id with gdown (or copy a local .zip path)."""
    if os.path.isfile(target):
        print("already downloaded:", target)
        return target
    if os.path.isfile(source):
        shutil.copy(source, target)
    else:
        gdown.download(source if source.startswith("http") else None,
                       target, id=None if source.startswith("http") else source,
                       quiet=False, fuzzy=True)
    if not zipfile.is_zipfile(target):
        raise RuntimeError(f"{target} is not a zip (is the Drive file shared as 'Anyone with the link'?)")
    return target

def unzip_to(zip_path, folder):
    if os.path.isdir(folder) and os.listdir(folder):
        print("already extracted:", folder)
        return
    os.makedirs(folder, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(folder)

def find_root(base, children):
    """Folder under ``base`` that directly contains any of ``children``
    (handles zips with or without a single top folder)."""
    for root, dirs, _ in os.walk(base):
        if any(c in dirs for c in children):
            return root
    raise FileNotFoundError(f"none of {children} found under {base}")

def is_set(source):
    return bool(source) and not source.startswith("PUT_")

os.makedirs(DATA_DIR, exist_ok=True)
unzip_to(fetch_zip(GT_ZIP, f"{DATA_DIR}/mot-gt.zip"), f"{DATA_DIR}/mot-gt")
GT_ROOT = find_root(f"{DATA_DIR}/mot-gt", DATASETS)

# Option A: one zip with all detector folders.
combined_parent = None
if is_set(DETECTIONS_ZIP):
    unzip_to(fetch_zip(DETECTIONS_ZIP, f"{DATA_DIR}/detections.zip"), f"{DATA_DIR}/detections")
    combined_parent = find_root(f"{DATA_DIR}/detections", DET_FOLDERS)

# Option B: one zip per detector folder (takes priority for that folder).
DET_ROOTS = []
for name in DET_FOLDERS:
    source = DETECTION_FOLDER_ZIPS.get(name, "")
    if is_set(source):
        folder = f"{DATA_DIR}/detections-{name}"
        unzip_to(fetch_zip(source, f"{DATA_DIR}/{name}.zip"), folder)
        DET_ROOTS.append(find_root(folder, DATASETS))  # the folder holding MOT17/, DanceTrack/, ...
    elif combined_parent and os.path.isdir(os.path.join(combined_parent, name)):
        DET_ROOTS.append(os.path.join(combined_parent, name))
    else:
        raise FileNotFoundError(f"no source for detector folder {name}: set DETECTIONS_ZIP "
                                f"or DETECTION_FOLDER_ZIPS['{name}']")

print("GT_ROOT   =", GT_ROOT)
print("DET_ROOTS =", DET_ROOTS)

In [ ]:
# What was found: sequences per split (with gt + seqinfo) and detection files per detector.
for name in DATASETS:
    for split in ("train", "val"):
        seqs = sorted(glob.glob(f"{GT_ROOT}/{name}/{split}/*/gt/gt.txt"))
        ok = [s for s in seqs if os.path.isfile(os.path.join(os.path.dirname(os.path.dirname(s)), "seqinfo.ini"))]
        if seqs:
            print(f"{name:10s} {split:5s}: {len(ok)} sequences")
for root in DET_ROOTS:
    if not os.path.isdir(root):
        print("MISSING detector folder:", root); continue
    counts = {n: len(glob.glob(f"{root}/{n}/*.txt")) for n in DATASETS if os.path.isdir(f"{root}/{n}")}
    print(os.path.basename(root), counts)

In [ ]:
# Fast unit tests (seconds): checks the training code before a long run.
!python -m pytest -q test_adaptive_kalman_contract.py test_paper_updates.py

## 3. Train

Every run aggregates all folders in `DET_FOLDERS`. Change `EPOCHS`/`EXTRA` as needed; any
`train_adaptive_kalman_real.py` flag can be added to `EXTRA`. Each run writes
`<save_dir>/best_model.pth` and `training_history.json`.

In [ ]:
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    CKPT_DIR = DRIVE_DIR
else:
    CKPT_DIR = f"{REPO_DIR}/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

EPOCHS = 60
COMMON = f"--data_root {GT_ROOT} --det_roots {' '.join(DET_ROOTS)} --epochs {EPOCHS}"
print(COMMON)
print("checkpoints ->", CKPT_DIR)

def train(name, flags=""):
    cmd = f"python train_adaptive_kalman_real.py {COMMON} {flags} --save_dir {CKPT_DIR}/{name}"
    print(cmd)
    get_ipython().system(cmd)

### 3a. Thesis model (5-term proxy loss), retrained on both detection sets

In [ ]:
train("thesis_proxy")

### 3b. Self-supervised innovation loss + learned velocity noise (U1 + U2)

In [ ]:
train("paper_innovation_q7", "--objective innovation --q_dim 7")

### 3c. Proxy + innovation loss (hybrid) + learned velocity noise

In [ ]:
train("paper_hybrid_q7", "--objective hybrid --q_dim 7")

### 3d. 3b + detection-context features (18-D input)

In [ ]:
train("paper_innovation_q7_ctx", "--objective innovation --q_dim 7 --use_detection_context")

If a run's loss becomes NaN, rerun it with `--no_innovation_kf_fp32` (float64 Kalman filter).
For several seeds (paper ablations) add `--seed 1`, `--seed 2`, ... with different names.

## 4. Summary and download

In [ ]:
import json
for path in sorted(glob.glob(f"{CKPT_DIR}/*/training_history.json")):
    hist = json.load(open(path))
    vals = hist.get("val_loss", [])
    if vals:
        best = min(range(len(vals)), key=vals.__getitem__)
        print(f"{os.path.basename(os.path.dirname(path)):28s} epochs {len(vals):3d} | best val {vals[best]:.4f} @ {best + 1}")

In [ ]:
# Zip all checkpoints for download (also kept in Drive when SAVE_TO_DRIVE).
shutil.make_archive("/content/checkpoints", "zip", CKPT_DIR)
from google.colab import files
files.download("/content/checkpoints.zip")